<a href="https://colab.research.google.com/github/JF11579/PSID_For_the_Rest_of_US/blob/main/Notebook_01_2_Prepare_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01.2 — Prepare Dataset (v2)

**Project:** PSID Book v2  
**Purpose:** Load and merge PSID extract + FIMS genealogy file; recode variables; apply sample restrictions; save clean intergenerational dataset.  
**Primary extract:** J358642.csv (774 vars)  
**FIMS file:** fim15712_gidpro_BO_2_BAL.xlsx  
**Last updated:** 2026-03-09

---
### Variable Reference
| Variable | Source | Description |
|----------|--------|-------------|
| ER30001 + ER30002 | PSID | person_id = (ER30001 * 1000) + ER30002 |
| ER30004 | PSID | Age in 1968 → birth_year = 1968 - ER30004 |
| ER32006 | PSID | Sample status (keep 1,2,3,4) |
| ER32000 | PSID | Sex (1=Male, 2=Female) |
| V103 | PSID | Parent homeownership (1=Owns → binary) |
| V313 | PSID | Parent education (categorical 0–8 → years) |
| V81 | PSID | Parent income 1968 |
| V181 | PSID | Race 1968 (1=White, 2=Black, 3=Hispanic, 7=Other) |
| V93 | PSID | State 1968 |
| V119 | PSID | Sex of 1968 Head |
| ER35152 | PSID | Child education years (0, 99 = missing) |
| ER82032 | PSID | Child homeownership 2023 (1=Owns → binary) |
| G1ID68, G1PN | FIMS | Parent ID components |
| G2ID68, G2PN | FIMS | Child ID components |

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Setup — Paths and Imports

In [ ]:
import pandas as pd
import numpy as np
import os

# ── Paths ──────────────────────────────────────────────────────────────────
#BASE       = '/content/drive/MyDrive/PSID/PSID_2026/PSID_book_v2'
BASE = '/content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2'
DATA_DIR   = os.path.join(BASE, 'data')
OUTPUT_DIR = os.path.join(BASE, 'outputs')

PSID_FILE  = os.path.join(DATA_DIR, 'J358642.csv')
#FIMS_FILE  = os.path.join(DATA_DIR, 'fim15712_gidpro_BO_2_BAL.xlsx')
FIMS_FILE  = os.path.join(DATA_DIR, 'fim15712_gidpro_BO_2_BAL.csv')
LABEL_FILE = os.path.join(DATA_DIR, 'J358642_labels.txt')

# ── Output filename ────────────────────────────────────────────────────────
OUT_FILE   = os.path.join(OUTPUT_DIR, 'psid_intergenerational_clean.csv')

# Confirm paths exist
for f in [PSID_FILE, FIMS_FILE]:
    status = '✅ Found' if os.path.exists(f) else '❌ NOT FOUND'
    print(f'{status}: {f}')

✅ Found: /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/data/J358642.csv
✅ Found: /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/data/fim15712_gidpro_BO_2_BAL.csv


## 2. Load PSID Extract (J358642)

In [ ]:
# All variables needed for v2 pipeline
# 'ER30003',   # age in 1968 (was ER30004 — not in this extract)
VARS_NEEDED = [
    # Identification
    'ER30001', 'ER30002',
    # Demographics
    'ER30004',   # age in 1968
    'ER32006',   # sample status
    'ER32000',   # sex
    # Parent (G1) variables — 1968 Family File
    'V103',      # parent homeownership
    'V313',      # parent education
    'V81',       # parent income 1968
    'V181',      # race 1968
    'V93',       # state 1968
    'V119',      # sex of 1968 head
    # Child (G2) outcome variables
    'ER35152',   # child education years
    'ER82032',   # child homeownership 2023
    'ER30003',   # age in 1968 (was ER30004 — not in this extract)
]

print('Loading PSID extract...')
psid_raw = pd.read_csv(PSID_FILE, usecols=VARS_NEEDED, low_memory=False)
print(f'Loaded: {psid_raw.shape[0]:,} rows × {psid_raw.shape[1]} columns')
psid_raw.head()

Loading PSID extract...
Loaded: 85,536 rows × 14 columns


,ER30001,ER30002,ER32000,ER32006,V81,V93,V103,V119,V181,V313,ER30003,ER30004,ER82032,ER35152
0,1,1,1,1,13108.0,24.0,1.0,1.0,1.0,2.0,1,52,NaN,0
1,1,2,2,1,13108.0,24.0,1.0,1.0,1.0,2.0,2,46,NaN,0
2,1,3,1,1,13108.0,24.0,1.0,1.0,1.0,2.0,7,21,NaN,0
3,1,4,2,1,13108.0,24.0,1.0,1.0,1.0,2.0,3,19,NaN,0
4,1,30,2,2,13108.0,24.0,1.0,1.0,1.0,2.0,0,0,NaN,0


In [ ]:
# Diagnostic: check what's actually in the columns
print("Shape:", psid.shape)
print("\nFirst 20 columns:", psid.columns[:20].tolist())
print("\nAll ER3xxxx columns:")
er3_cols = [c for c in psid.columns if str(c).startswith('ER3')]
print(er3_cols)

# Check for ER32006 specifically with fuzzy match
matches = [c for c in psid.columns if '32006' in str(c)]
print("\nAnything containing '32006':", matches)

Shape: (54432, 16)

First 20 columns: ['ER30001', 'ER30002', 'ER32000', 'ER32006', 'V81', 'V93', 'V103', 'V119', 'V181', 'V313', 'ER30003', 'ER30004', 'ER82032', 'ER35152', 'person_id', 'birth_year']

All ER3xxxx columns:
['ER30001', 'ER30002', 'ER32000', 'ER32006', 'ER30003', 'ER30004', 'ER35152']

Anything containing '32006': ['ER32006']


## 3. Create person_id and birth_year

In [ ]:
psid = psid_raw.copy()

# person_id: standard PSID construction
psid['person_id'] = (psid['ER30001'] * 1000) + psid['ER30002']

# birth_year derived from age in 1968
# ER30004 = 0 means age unknown — treat as missing
psid['birth_year'] = np.where(psid['ER30004'] > 0,
                               1968 - psid['ER30004'],
                               np.nan)

print(f'person_id created. Unique IDs: {psid["person_id"].nunique():,}')
print(f'birth_year range: {psid["birth_year"].min():.0f} – {psid["birth_year"].max():.0f}')
print(f'birth_year missing: {psid["birth_year"].isna().sum():,}')

person_id created. Unique IDs: 85,536
birth_year range: 969 – 1967
birth_year missing: 67,306


## 4. Filter to Sample Members (ER32006)

In [ ]:
# Keep only original PSID sample members
# ER32006: 1=SRC original, 2=Census SEO, 3=Latino 1990, 4=Immigrant 1997
VALID_SAMPLE = [1, 2, 3, 4]

n_before = len(psid)
psid = psid[psid['ER32006'].isin(VALID_SAMPLE)].copy()
n_after = len(psid)

print(f'Before sample filter: {n_before:,}')
print(f'After sample filter:  {n_after:,}')
print(f'Dropped: {n_before - n_after:,}')
print(f'\nSample breakdown:\n{psid["ER32006"].value_counts().sort_index()}')

Before sample filter: 85,536
After sample filter:  54,432
Dropped: 31,104

Sample breakdown:
ER32006
1    30139
2    21505
3     2684
4      104
Name: count, dtype: int64


## 5. Load and Merge FIMS Genealogy File

In [ ]:
print('Loading FIMS file...')
fims_raw = pd.read_csv(FIMS_FILE)
print(f'FIMS loaded: {fims_raw.shape[0]:,} rows × {fims_raw.shape[1]} columns')
print(f'FIMS columns: {list(fims_raw.columns)}')
fims_raw.head()

Loading FIMS file...
FIMS loaded: 58,585 rows × 8 columns
FIMS columns: ['G1ID68', 'G1PN', 'G1TYPE', 'G1POS', 'G2ID68', 'G2PN', 'G2TYPE', 'G2POS']


,G1ID68,G1PN,G1TYPE,G1POS,G2ID68,G2PN,G2TYPE,G2POS
0,2,2,M,1,2,171,F,2
1,2,2,M,1,2,172,I,2
2,2,2,M,1,2,902,I,2
3,2,2,M,1,2,903,I,2
4,2,2,M,1,2,904,I,2


In [ ]:
fims = fims_raw.copy()

# Construct parent_id and child_id using standard PSID construction
fims['parent_id'] = (fims['G1ID68'] * 1000) + fims['G1PN']
fims['child_id']  = (fims['G2ID68'] * 1000) + fims['G2PN']

# Keep only the linkage columns
fims = fims[['parent_id', 'child_id']].drop_duplicates()

print(f'FIMS linkage pairs: {len(fims):,}')
print(f'Unique parents:     {fims["parent_id"].nunique():,}')
print(f'Unique children:    {fims["child_id"].nunique():,}')

FIMS linkage pairs: 58,585
Unique parents:     21,250
Unique children:    47,414


In [ ]:
# Separate parent-level and child-level data from PSID
# Parent vars: the 1968 family-level variables (V103, V313, V81, V181, V93, V119)
# These are attached to parent person_id

PARENT_VARS = ['person_id', 'V103', 'V313', 'V81', 'V181', 'V93', 'V119']
CHILD_VARS  = ['person_id', 'birth_year', 'ER32000', 'ER35152', 'ER82032']

psid_parents  = psid[PARENT_VARS].copy().rename(columns={'person_id': 'parent_id'})
psid_children = psid[CHILD_VARS].copy().rename(columns={'person_id': 'child_id'})

print(f'Parent records: {len(psid_parents):,}')
print(f'Child records:  {len(psid_children):,}')

Parent records: 54,432
Child records:  54,432


In [ ]:
# Merge: FIMS → parent vars → child vars
df = fims.merge(psid_parents, on='parent_id', how='inner')
print(f'After merging parent vars: {len(df):,}')

df = df.merge(psid_children, on='child_id', how='inner')
print(f'After merging child vars:  {len(df):,}')

df.head()

After merging parent vars: 57,982
After merging child vars:  45,138


,parent_id,child_id,V103,V313,V81,V181,V93,V119,birth_year,ER32000,ER35152,ER82032
0,4001,4003,1.0,2.0,6434.0,1.0,41.0,1.0,1952.0,1,0,NaN
1,4001,4004,1.0,2.0,6434.0,1.0,41.0,1.0,1954.0,2,0,NaN
2,4001,4005,1.0,2.0,6434.0,1.0,41.0,1.0,1956.0,2,0,NaN
3,4001,4006,1.0,2.0,6434.0,1.0,41.0,1.0,1958.0,2,0,NaN
4,4001,4007,1.0,2.0,6434.0,1.0,41.0,1.0,1961.0,2,12,1.0


## 6. Recode Variables

In [ ]:
# ── V103: Parent Homeownership → binary ───────────────────────────────────
# 1=Owns, 5=Rents, 8=Neither; 0/9 = missing
df['parent_homeowner'] = np.where(df['V103'] == 1, 1,
                          np.where(df['V103'].isin([5, 8]), 0, np.nan))

print('parent_homeowner:')
print(df['parent_homeowner'].value_counts(dropna=False))

parent_homeowner:
parent_homeowner
1.0    16831
0.0    16824
NaN    11483
Name: count, dtype: int64


In [ ]:
# ── V313: Parent Education → approximate years ────────────────────────────
ed_map = {
    0: 3,   # <5th grade, difficulty reading
    1: 3,   # <5th grade, no difficulty
    2: 7,   # 6–8th grade
    3: 10,  # 9–11th grade
    4: 12,  # 12th grade / HS complete
    5: 13,  # 12th + non-academic training
    6: 14,  # College, no degree
    7: 16,  # BA/BS
    8: 18,  # Advanced/professional degree
    9: np.nan  # NA/DK — drop
}
df['parent_educ_yrs'] = df['V313'].map(ed_map)

print('parent_educ_yrs:')
print(df['parent_educ_yrs'].value_counts(dropna=False).sort_index())

parent_educ_yrs:
parent_educ_yrs
3.0      4249
7.0      7251
10.0     7975
12.0     5339
13.0     2664
14.0     3134
16.0     1852
18.0      991
NaN     11683
Name: count, dtype: int64


In [ ]:
# ── V81: Parent Income 1968 ───────────────────────────────────────────────
# Treat 0 and negative as missing; keep as continuous
# Optional log transform — create both versions
df['parent_income_1968'] = np.where(df['V81'] > 0, df['V81'], np.nan)
df['parent_income_1968_log'] = np.log(df['parent_income_1968'])

print('parent_income_1968 — descriptive stats:')
print(df['parent_income_1968'].describe())

parent_income_1968 — descriptive stats:
count    33648.000000
mean      7600.013552
std       5545.377111
min        135.000000
25%       3900.000000
50%       6313.000000
75%      10000.000000
max      65400.000000
Name: parent_income_1968, dtype: float64


In [ ]:
# ── V181: Race 1968 → labeled categories ─────────────────────────────────
# 1=White, 2=Black, 3=Hispanic, 7=Other; 0/9 = missing
race_map = {1: 'White', 2: 'Black', 3: 'Hispanic', 7: 'Other'}
df['race'] = df['V181'].map(race_map)  # unmapped codes → NaN

print('race:')
print(df['race'].value_counts(dropna=False))

race:
race
White       17945
Black       14892
NaN         11546
Hispanic      549
Other         206
Name: count, dtype: int64


In [ ]:
# ── V93: State 1968 → keep as numeric PSID state code ────────────────────
# (PSID uses its own state coding; recode to FIPS later if needed)
df['state_1968'] = np.where(df['V93'] > 0, df['V93'], np.nan)

# ── V119: Sex of 1968 Head ────────────────────────────────────────────────
df['head_sex_1968'] = df['V119']  # keep as-is; 1=Male, 2=Female

# ── ER32000: Child sex ────────────────────────────────────────────────────
df['child_sex'] = df['ER32000']  # 1=Male, 2=Female

print('state_1968 non-missing:', df['state_1968'].notna().sum())
print('child_sex:'); print(df['child_sex'].value_counts(dropna=False))

state_1968 non-missing: 33655
child_sex:
child_sex
1    22911
2    22227
Name: count, dtype: int64


In [ ]:
# ── ER35152: Child Education Years ───────────────────────────────────────
# 1–17 = actual years; 0 and 99 = missing
df['child_educ_yrs'] = np.where(df['ER35152'].isin([0, 99]),
                                 np.nan,
                                 df['ER35152'])

print('child_educ_yrs — descriptive stats:')
print(df['child_educ_yrs'].describe())

child_educ_yrs — descriptive stats:
count    12484.000000
mean        13.546940
std          2.243231
min          4.000000
25%         12.000000
50%         13.000000
75%         16.000000
max         17.000000
Name: child_educ_yrs, dtype: float64


In [ ]:
# ── ER82032: Child Homeownership 2023 → binary ───────────────────────────
# 1=Owns, 5=Rents, 8=Neither; 0/9 = missing
df['child_homeowner_2023'] = np.where(df['ER82032'] == 1, 1,
                              np.where(df['ER82032'].isin([5, 8]), 0, np.nan))

print('child_homeowner_2023:')
print(df['child_homeowner_2023'].value_counts(dropna=False))

child_homeowner_2023:
child_homeowner_2023
NaN    26289
1.0    10567
0.0     8282
Name: count, dtype: int64


## 7. Age / Education / Sample Restrictions

In [ ]:
# Restrict to children aged ≥25 in 2023
# birth_year ≤ 1998 → age ≥ 25 in 2023
n_before = len(df)
df = df[df['birth_year'] <= 1998].copy()
n_after = len(df)

print(f'Before age restriction (≥25 in 2023): {n_before:,}')
print(f'After age restriction:                {n_after:,}')
print(f'Dropped: {n_before - n_after:,}')

Before age restriction (≥25 in 2023): 45,138
After age restriction:                11,664
Dropped: 33,474


In [ ]:
# Optional: restrict to children with non-missing education AND homeownership outcome
# (comment out if you want to retain partial cases for different analyses)

n_before = len(df)
df_analysis = df[
    df['child_educ_yrs'].notna() &
    df['child_homeowner_2023'].notna() &
    df['parent_homeowner'].notna()
].copy()
n_after = len(df_analysis)

print(f'Before outcome missingness drop: {n_before:,}')
print(f'After outcome missingness drop:  {n_after:,}')
print(f'Dropped: {n_before - n_after:,}')

print(f'\n--- FINAL ANALYTIC SAMPLE: {n_after:,} observations ---')

Before outcome missingness drop: 11,664
After outcome missingness drop:  3,560
Dropped: 8,104

--- FINAL ANALYTIC SAMPLE: 3,560 observations ---


## 8. Select and Label Final Columns

In [ ]:
# Check what columns are actually in the file
psid_check = pd.read_csv(PSID_FILE, nrows=5)
print(psid_check.columns.tolist())

['ER30000', 'ER30001', 'ER30002', 'ER32000', 'ER32004', 'ER32006', 'ER32009', 'V1', 'V5', 'V81', 'V93', 'V103', 'V104', 'V107', 'V117', 'V119', 'V181', 'V313', 'V314', 'ER30003', 'ER30004', 'V441', 'V449', 'V593', 'V594', 'V597', 'V793', 'V794', 'V795', 'V824', 'V1008', 'V1010', 'ER30020', 'ER30021', 'ER30022', 'ER30023', 'V1101', 'V1122', 'V1239', 'V1240', 'V1264', 'V1265', 'V1268', 'V1484', 'V1486', 'V1511', 'ER30043', 'ER30044', 'ER30045', 'ER30046', 'V1801', 'V1823', 'V1942', 'V1943', 'V1967', 'V1968', 'V1973', 'V2197', 'V2198', 'V2223', 'ER30067', 'ER30068', 'ER30069', 'ER30070', 'V2401', 'V2423', 'V2542', 'V2543', 'V2566', 'V2567', 'V2823', 'V2824', 'V2849', 'ER30091', 'ER30092', 'ER30093', 'ER30094', 'V3001', 'V3021', 'V3095', 'V3096', 'V3108', 'V3241', 'V3242', 'V3255', 'ER30117', 'ER30118', 'ER30119', 'ER30120', 'V3401', 'V3417', 'V3508', 'V3509', 'V3522', 'V3634', 'V3662', 'V3663', 'V3664', 'V3675', 'ER30138', 'ER30139', 'ER30140', 'ER30141', 'V3801', 'V3817', 'V3921', 'V3922

In [ ]:
psid_check = pd.read_csv(PSID_FILE, nrows=5)
actual_cols = set(psid_check.columns.tolist())

needed = ['ER30001', 'ER30002', 'ER30003', 'ER32006', 'ER32000',
          'V103', 'V313', 'V81', 'V181', 'V93', 'V119',
          'ER35152', 'ER82032']

print('MISSING from file:')
for v in needed:
    if v not in actual_cols:
        print(f'  ❌ {v}')

print('\nPRESENT in file:')
for v in needed:
    if v in actual_cols:
        print(f'  ✅ {v}')

MISSING from file:

PRESENT in file:
  ✅ ER30001
  ✅ ER30002
  ✅ ER30003
  ✅ ER32006
  ✅ ER32000
  ✅ V103
  ✅ V313
  ✅ V81
  ✅ V181
  ✅ V93
  ✅ V119
  ✅ ER35152
  ✅ ER82032


In [ ]:
FINAL_COLS = [
    # IDs
    'child_id', 'parent_id',
    # Child demographics
    'birth_year', 'child_sex',
    # Child outcomes
    'child_educ_yrs', 'child_homeowner_2023',
    # Parent characteristics (G1)
    'parent_homeowner', 'parent_educ_yrs',
    'parent_income_1968', 'parent_income_1968_log',
    'race', 'state_1968', 'head_sex_1968',
    # Raw source vars retained for checking
    'V103', 'V313', 'V81', 'V181', 'V93', 'V119',
    'ER35152', 'ER82032', 'ER32000'
]

df_final = df_analysis[FINAL_COLS].copy()

print(f'Final dataset: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns')
df_final.head()

Final dataset: 3,560 rows × 22 columns


,child_id,parent_id,birth_year,child_sex,child_educ_yrs,child_homeowner_2023,parent_homeowner,parent_educ_yrs,parent_income_1968,parent_income_1968_log,...,head_sex_1968,V103,V313,V81,V181,V93,V119,ER35152,ER82032,ER32000
4,4007,4001,1961.0,2,12.0,1.0,1.0,7.0,6434.0,8.769352,...,1.0,1.0,2.0,6434.0,1.0,41.0,1.0,12,1.0,2
10,4007,4002,1961.0,2,12.0,1.0,1.0,7.0,6434.0,8.769352,...,1.0,1.0,2.0,6434.0,1.0,41.0,1.0,12,1.0,2
42,5003,5001,1963.0,1,12.0,1.0,1.0,10.0,7900.0,8.974618,...,1.0,1.0,3.0,7900.0,1.0,41.0,1.0,12,1.0,1
43,5004,5001,1966.0,2,12.0,0.0,1.0,10.0,7900.0,8.974618,...,1.0,1.0,3.0,7900.0,1.0,41.0,1.0,12,5.0,2
44,5005,5001,1967.0,1,10.0,1.0,1.0,10.0,7900.0,8.974618,...,1.0,1.0,3.0,7900.0,1.0,41.0,1.0,10,1.0,1


## 9. Descriptive Check

In [ ]:
print('=== DESCRIPTIVE SUMMARY — KEY VARIABLES ===')
print()

key_vars = [
    'parent_homeowner', 'parent_educ_yrs', 'parent_income_1968',
    'child_educ_yrs', 'child_homeowner_2023', 'birth_year'
]
print(df_final[key_vars].describe().round(2))

print()
print('--- Parent homeownership rate ---')
print(f"{df_final['parent_homeowner'].mean():.1%}")

print()
print('--- Child homeownership rate (2023) ---')
print(f"{df_final['child_homeowner_2023'].mean():.1%}")

print()
print('--- Race distribution ---')
print(df_final['race'].value_counts(dropna=False))

=== DESCRIPTIVE SUMMARY — KEY VARIABLES ===

       parent_homeowner  parent_educ_yrs  parent_income_1968  child_educ_yrs  \
count           3560.00          3528.00             3560.00         3560.00   
mean               0.62            10.65             9257.13           13.78   
std                0.48             3.96             6572.44            2.17   
min                0.00             3.00              275.00            4.00   
25%                0.00             7.00             4802.25           12.00   
50%                1.00            12.00             8075.00           14.00   
75%                1.00            13.00            12060.00           16.00   
max                1.00            18.00            65400.00           17.00   

       child_homeowner_2023  birth_year  
count               3560.00     3560.00  
mean                   0.73     1957.94  
std                    0.45        6.02  
min                    0.00     1934.00  
25%                    0

## 10. Save Output

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_final.to_csv(OUT_FILE, index=False)

print(f'✅ Saved: {OUT_FILE}')
print(f'   Rows: {len(df_final):,}')
print(f'   Columns: {df_final.shape[1]}')

# Also save the pre-missingness-drop version for robustness checks
out_full = OUT_FILE.replace('.csv', '_prefilt.csv')
df[FINAL_COLS].to_csv(out_full, index=False)
print(f'✅ Saved (pre-filter): {out_full}')

✅ Saved: /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/outputs/psid_intergenerational_clean.csv
   Rows: 3,560
   Columns: 22
✅ Saved (pre-filter): /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/outputs/psid_intergenerational_clean_prefilt.csv


---
## End of Notebook 01.2
**Next:** Notebook 02 — Main Analysis
- Baseline regression: `child_educ_yrs ~ parent_homeowner + controls`
- Homeownership outcome: `child_homeowner_2023 ~ parent_homeowner + controls`
- Tables and figures for book